# Medalian Architechture

In [0]:
data = [
    (1,"Ram","Laptop",1200),
    (2,"Sam","Mobile",800),
    (3,None,"Tablet",500),
    (4,"John","Laptop",-100),
    (5,"Ram","Mobile",800),
    (5,"Ram","Mobile",800)
]

columns = ["id","customer","product","amount"]

df = spark.createDataFrame(data,columns)

df.show()


In [0]:
bronze_path = "/Volumes/workspace/delta/medallion/bronze/sales"

df.write \
.format("delta") \
.mode("overwrite") \
.save(bronze_path)


In [0]:
bronze_table=spark.read.format("delta").load(bronze_path)
bronze_table.show()

In [0]:
silver_table = bronze_table.dropDuplicates()
silver_table = silver_table.fillna({"customer":"unknown"})
silver_table = silver_table.filter("amount > 0")
silver_table = silver_table.withColumnRenamed("id","sale_id")


In [0]:
display(silver_table)

In [0]:
gold_table = silver_table.groupBy("product").agg({"amount":"sum"})
gold_table = gold_table.withColumnRenamed("sum(amount)","total_sales")
gold_table = gold_table.orderBy("total_sales",ascending=False)
display(gold_table)

In [0]:
silver_path = "/Volumes/workspace/delta/medallion/silver"


silver_table.write \
.format("delta") \
.mode("overwrite") \
.save(silver_path)



In [0]:
silver_table = spark.read.format("delta").load(silver_path)

In [0]:
display(silver_table)

In [0]:
gold_path = "/Volumes/workspace/delta/medallion/gold"

In [0]:
gold_table.write.format("delta").mode("overwrite").save(gold_path)
gold_table = spark.read.format("delta").load(gold_path)
display(gold_table)

In [0]:
from delta.tables import DeltaTable
bronze_table = DeltaTable.forPath(spark, bronze_path)
display(bronze_table.history())

In [0]:
spark.read \
.format("delta") \
.option("timestampAsOf","2026-03-10T13:10:14.000+00:00") \
.load(bronze_path) \
.show()


In [0]:
display(dbutils.fs.ls(bronze_path))
